In [2]:
import sys
print(sys.executable)


C:\Users\manov\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe


In [1]:
%pip install pyarrow
%pip install fastparquet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\manov\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\manov\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from fastparquet import write
import pandas as pd     
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os


In [17]:
import pandas as pd

# Windows-style path, be sure to use r-string or double backslashes
file_path = r"C:\Users\manov\OneDrive\The Quant Desk\kan_mega_merged_part.0.parquet"

# Load it
df = pd.read_parquet(file_path)

# Quick checks
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns[:10]} ... {df.columns[-10:]}")
print(df.head(3))
print(df.tail(3))
print(df.info())

Shape: (5114, 284)
Columns: Index(['Date', 'PSJ_daily_Close', 'PSJ_daily_High', 'PSJ_daily_Low',
       'PSJ_daily_Open', 'PSJ_daily_Volume', 'USD_Index_DEXUSAL',
       'SOXL_daily_Close', 'SOXL_daily_High', 'SOXL_daily_Low'],
      dtype='object') ... Index(['PNQI_daily_Low', 'PNQI_daily_Open', 'PNQI_daily_Volume',
       'EDOC_daily_Close', 'EDOC_daily_High', 'EDOC_daily_Low',
       'EDOC_daily_Open', 'EDOC_daily_Volume', '30Y_Treasury_DGS30',
       'WTI_Oil_DCOILWTICO'],
      dtype='object')
                          Date  PSJ_daily_Close  PSJ_daily_High  \
__null_dask_index__                                               
0                   2010-01-01              NaN             NaN   
1                   2010-01-02              NaN             NaN   
2                   2010-01-03              NaN             NaN   

                     PSJ_daily_Low  PSJ_daily_Open  PSJ_daily_Volume  \
__null_dask_index__                                                    
0               

In [4]:
df = df.reset_index(drop=True)


In [5]:
print(df['Date'].min(), df['Date'].max())

2010-01-01 00:00:00 2024-01-01 00:00:00


In [6]:
print(df.columns[df.isnull().mean() < 0.95])  # Columns with at least 5% non-NaN


Index(['Date', 'PSJ_daily_Close', 'PSJ_daily_High', 'PSJ_daily_Low',
       'PSJ_daily_Open', 'PSJ_daily_Volume', 'USD_Index_DEXUSAL',
       'SOXL_daily_Close', 'SOXL_daily_High', 'SOXL_daily_Low',
       ...
       'PNQI_daily_Low', 'PNQI_daily_Open', 'PNQI_daily_Volume',
       'EDOC_daily_Close', 'EDOC_daily_High', 'EDOC_daily_Low',
       'EDOC_daily_Open', 'EDOC_daily_Volume', '30Y_Treasury_DGS30',
       'WTI_Oil_DCOILWTICO'],
      dtype='object', length=273)


In [7]:
print(df.columns[df.isnull().all()])  # All-NaN columns


Index([], dtype='object')


In [10]:
import pandas as pd

# Replace 'Date' with your actual date column name if it's different
value_counts = pd.DataFrame({
    "non_NaN_count": df.drop('Date', axis=1).notna().sum(),
    "NaN_count": df.drop('Date', axis=1).isna().sum(),
    "non_NaN_pct": df.drop('Date', axis=1).notna().mean() * 100
}).sort_values("non_NaN_count", ascending=False)

pd.set_option('display.max_rows', 20)  # Show more or fewer rows as needed
print(value_counts.head(20))           # Top 20 columns with the most data
print("\n...lowest coverage columns:")
print(value_counts.tail(10))           # Bottom 10 columns with least data

# Summary stats for the whole DataFrame
print("\n---- OVERALL COVERAGE ----")
print("Mean non-NaN % across all columns: {:.1f}%".format(value_counts['non_NaN_pct'].mean()))
print("Median non-NaN % across all columns: {:.1f}%".format(value_counts['non_NaN_pct'].median()))


                                        non_NaN_count  NaN_count  non_NaN_pct
Economic_Policy_Uncertainty_USEPUINDXD           5114          0   100.000000
HY_Spread_BAMLH0A0HYM2                           3654       1460    71.450919
VIXCLS_VIXCLS                                    3534       1580    69.104419
VXNCLS_VXNCLS                                    3524       1590    68.908878
NASDAQ_NASDAQCOM                                 3523       1591    68.889323
PSJ_daily_Close                                  3522       1592    68.869769
TECL_daily_Volume                                3522       1592    68.869769
TECL_daily_Open                                  3522       1592    68.869769
PSJ_daily_Low                                    3522       1592    68.869769
PNQI_daily_Volume                                3522       1592    68.869769
TECL_daily_High                                  3522       1592    68.869769
TECL_daily_Close                                 3522       1592

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Load your parquet file (update path as needed)
file_path = r"C:\Users\manov\OneDrive\The Quant Desk\kan_mega_merged_part.0.parquet"
df = pd.read_parquet(file_path)

# Backup the original
df_orig = df.copy()


# Frequency-Aware Forward-Fill for Macro Variables
First, I'll define which macro columns are monthly and quarterly.

In [3]:
# Print all columns (just to see what I'm working with)
print(df.columns.tolist())

# Define some typical macro keywords 
macro_keywords = [
    'CPI', 'PPI', 'GDP', 'Unemployment', 'Payroll', 'M2', 'Housing', 'Median_House', 'Fed_Funds',
    'Industrial_Production', 'Total_Loans', 'Treasury', 'Spread', 'Economic_Policy', 'VIX', 'VXNCLS'
]

# Find columns matching any macro keyword
macro_cols = [col for col in df.columns if any(kw.lower() in col.lower() for kw in macro_keywords) and not col.endswith('_missing')]

print("\n==== POSSIBLE MACRO COLUMNS ====")
for col in macro_cols:
    print(col)


['Date', 'PSJ_daily_Close', 'PSJ_daily_High', 'PSJ_daily_Low', 'PSJ_daily_Open', 'PSJ_daily_Volume', 'USD_Index_DEXUSAL', 'SOXL_daily_Close', 'SOXL_daily_High', 'SOXL_daily_Low', 'SOXL_daily_Open', 'SOXL_daily_Volume', 'IGV_daily_Close', 'IGV_daily_High', 'IGV_daily_Low', 'IGV_daily_Open', 'IGV_daily_Volume', 'GDP_GDPC1', 'FINX_daily_Close', 'FINX_daily_High', 'FINX_daily_Low', 'FINX_daily_Open', 'FINX_daily_Volume', 'HY_Spread_BAMLH0A0HYM2', 'ARKW_daily_Close', 'ARKW_daily_High', 'ARKW_daily_Low', 'ARKW_daily_Open', 'ARKW_daily_Volume', 'M2_M2SL', 'USD_EUR_DEXUSEU', 'DJIA_DJIA', 'IDRV_daily_Close', 'IDRV_daily_High', 'IDRV_daily_Low', 'IDRV_daily_Open', 'IDRV_daily_Volume', 'TDIV_daily_Close', 'TDIV_daily_High', 'TDIV_daily_Low', 'TDIV_daily_Open', 'TDIV_daily_Volume', 'DRIV_daily_Close', 'DRIV_daily_High', 'DRIV_daily_Low', 'DRIV_daily_Open', 'DRIV_daily_Volume', 'HACK_daily_Close', 'HACK_daily_High', 'HACK_daily_Low', 'HACK_daily_Open', 'HACK_daily_Volume', 'XLU_daily_Close', 'XLU_d

In [4]:
macro_cols = [
    "GDP_GDPC1", "HY_Spread_BAMLH0A0HYM2", "M2_M2SL", "2Y_Treasury_DGS2",
    "Core_CPI_CPILFESL", "Housing_Starts_HOUST", "10Y_Treasury_DGS10",
    "TED_Spread_TEDRATE", "CPI_CPIAUCSL", "PPI_PPIACO",
    "Economic_Policy_Uncertainty_USEPUINDXD", "VXNCLS_VXNCLS", "BAA10Y_Spread_BAA10Y",
    "Payrolls_PAYEMS", "Industrial_Production_INDPRO", "5Y_Treasury_DGS5",
    "Median_House_Price_MSPUS", "1Y_Treasury_DGS1", "Unemployment_UNRATE",
    "Fed_Funds_FEDFUNDS", "VIXCLS_VIXCLS", "Total_Loans_TOTLL", "30Y_Treasury_DGS30"
]

# 1. Add Missingness Indicators & Forward Fill All Macro Columns 

Missingness Indicators:

Helps the model learn that “imputed” data might be lower quality.

Captures regime changes or data glitches that may *themselves* be predictive.

Provides a way for the KAN  model to “hedge” or discount signals on uncertain days


In [5]:
df_macro = df.copy()

# 3. ADD MISSINGNESS INDICATORS **BEFORE** FILLING
for col in macro_cols:
    ind_col = f"{col}_imputed"
    df_macro[ind_col] = df_macro[col].isna().astype(int)


In [6]:
# Forward fill ALL macro columns (works great for monthly/quarterly releases)
df_macro[macro_cols] = df_macro[macro_cols].ffill()
# Check if any macro columns still have NaNs
print("\n==== MACRO COLUMNS AFTER FORWARD-FILL ====")   


==== MACRO COLUMNS AFTER FORWARD-FILL ====


In [7]:
print(df_macro[macro_cols].isna().sum())
print(df_macro.head(3))
print(df_macro[[f"{col}_imputed" for col in macro_cols]].head(3))

GDP_GDPC1                                 0
HY_Spread_BAMLH0A0HYM2                    3
M2_M2SL                                   0
2Y_Treasury_DGS2                          3
Core_CPI_CPILFESL                         0
Housing_Starts_HOUST                      0
10Y_Treasury_DGS10                        3
TED_Spread_TEDRATE                        3
CPI_CPIAUCSL                              0
PPI_PPIACO                                0
Economic_Policy_Uncertainty_USEPUINDXD    0
VXNCLS_VXNCLS                             3
BAA10Y_Spread_BAA10Y                      3
Payrolls_PAYEMS                           0
Industrial_Production_INDPRO              0
5Y_Treasury_DGS5                          3
Median_House_Price_MSPUS                  0
1Y_Treasury_DGS1                          3
Unemployment_UNRATE                       0
Fed_Funds_FEDFUNDS                        0
VIXCLS_VIXCLS                             3
Total_Loans_TOTLL                         5
30Y_Treasury_DGS30              

# 2. Time-based linear interpolation for any remaining gaps

This fills “within-release” gaps, making the macro data smooth and continuous—like good way to treat noisy macro time series.

Only interpolate where data exists on both sides—no lookahead

Set 'Date' column as the index before running interpolation step.

In [8]:
# If your date column is not already datetime:
df_macro['Date'] = pd.to_datetime(df_macro['Date'], errors='coerce')
df_macro = df_macro.set_index('Date')
for col in macro_cols:
    df_macro[col] = df_macro[col].interpolate(method='time', limit_direction='forward')

Now move it back

In [9]:
df_macro = df_macro.reset_index()

# Check It

Are Any Macro Columns Still Missing Data?

In [10]:
print("NaN count in macro columns (should be 0 or very few):")
print(df_macro[macro_cols].isna().sum())

NaN count in macro columns (should be 0 or very few):
GDP_GDPC1                                 0
HY_Spread_BAMLH0A0HYM2                    3
M2_M2SL                                   0
2Y_Treasury_DGS2                          3
Core_CPI_CPILFESL                         0
Housing_Starts_HOUST                      0
10Y_Treasury_DGS10                        3
TED_Spread_TEDRATE                        3
CPI_CPIAUCSL                              0
PPI_PPIACO                                0
Economic_Policy_Uncertainty_USEPUINDXD    0
VXNCLS_VXNCLS                             3
BAA10Y_Spread_BAA10Y                      3
Payrolls_PAYEMS                           0
Industrial_Production_INDPRO              0
5Y_Treasury_DGS5                          3
Median_House_Price_MSPUS                  0
1Y_Treasury_DGS1                          3
Unemployment_UNRATE                       0
Fed_Funds_FEDFUNDS                        0
VIXCLS_VIXCLS                             3
Total_Loans_TOTLL     

Do Missingness Indicators Match the Original Missing Data?

For each macro column, the indicator should be 1 if the original value was NaN, 0 if observed, regardless of filling.



In [11]:
for col in macro_cols:
    indicator_col = f"{col}_imputed"
    # Should be zero: count of rows where indicator ≠ (originally NaN)
    mismatches = ((df_orig[col].isna()) != (df_macro[indicator_col] == 1)).sum()
    print(f"Mismatches for {col}: {mismatches}")

Mismatches for GDP_GDPC1: 0
Mismatches for HY_Spread_BAMLH0A0HYM2: 0
Mismatches for M2_M2SL: 0
Mismatches for 2Y_Treasury_DGS2: 0
Mismatches for Core_CPI_CPILFESL: 0
Mismatches for Housing_Starts_HOUST: 0
Mismatches for 10Y_Treasury_DGS10: 0
Mismatches for TED_Spread_TEDRATE: 0
Mismatches for CPI_CPIAUCSL: 0
Mismatches for PPI_PPIACO: 0
Mismatches for Economic_Policy_Uncertainty_USEPUINDXD: 0
Mismatches for VXNCLS_VXNCLS: 0
Mismatches for BAA10Y_Spread_BAA10Y: 0
Mismatches for Payrolls_PAYEMS: 0
Mismatches for Industrial_Production_INDPRO: 0
Mismatches for 5Y_Treasury_DGS5: 0
Mismatches for Median_House_Price_MSPUS: 0
Mismatches for 1Y_Treasury_DGS1: 0
Mismatches for Unemployment_UNRATE: 0
Mismatches for Fed_Funds_FEDFUNDS: 0
Mismatches for VIXCLS_VIXCLS: 0
Mismatches for Total_Loans_TOTLL: 0
Mismatches for 30Y_Treasury_DGS30: 0


Spot-Check: Are Imputed Values Really Imputed?

For a few macro columns, print the date, original value, filled value, and indicator, where the indicator is 1:

In [12]:
for col in macro_cols[:3]:  # Change [:3] to any range or specific column names for deeper checks
    indicator_col = f"{col}_imputed"
    print(f"\n==== Spot check for: {col} ====")
    # Show rows where value was imputed
    check = df_macro.loc[df_macro[indicator_col] == 1, ['Date', col, indicator_col]]
    print(check.head(10))


==== Spot check for: GDP_GDPC1 ====
         Date  GDP_GDPC1  GDP_GDPC1_imputed
1  2010-01-02   16582.71                  1
2  2010-01-03   16582.71                  1
3  2010-01-04   16582.71                  1
4  2010-01-05   16582.71                  1
5  2010-01-06   16582.71                  1
6  2010-01-07   16582.71                  1
7  2010-01-08   16582.71                  1
8  2010-01-09   16582.71                  1
9  2010-01-10   16582.71                  1
10 2010-01-11   16582.71                  1

==== Spot check for: HY_Spread_BAMLH0A0HYM2 ====
         Date  HY_Spread_BAMLH0A0HYM2  HY_Spread_BAMLH0A0HYM2_imputed
0  2010-01-01                     NaN                               1
1  2010-01-02                     NaN                               1
2  2010-01-03                     NaN                               1
8  2010-01-09                    6.02                               1
9  2010-01-10                    6.02                               1
15 2010-0

Coverage Summary after Filling

In [13]:
print("\nMean non-NaN % across all macro cols:", 
      (df_macro[macro_cols].notna().mean() * 100).mean())
print("Median non-NaN % across all macro cols:", 
      (df_macro[macro_cols].notna().mean() * 100).median())



Mean non-NaN % across all macro cols: 99.97024366189997
Median non-NaN % across all macro cols: 100.0


Check That Indicators Did NOT Change After Filling

In [14]:
# These should still match the original, not the filled DataFrame
print("First few indicator columns (should be 1 for originally missing, 0 otherwise):")
print(df_macro[[f"{col}_imputed" for col in macro_cols]].head(10))


First few indicator columns (should be 1 for originally missing, 0 otherwise):
   GDP_GDPC1_imputed  HY_Spread_BAMLH0A0HYM2_imputed  M2_M2SL_imputed  \
0                  0                               1                0   
1                  1                               1                1   
2                  1                               1                1   
3                  1                               0                1   
4                  1                               0                1   
5                  1                               0                1   
6                  1                               0                1   
7                  1                               0                1   
8                  1                               1                1   
9                  1                               1                1   

   2Y_Treasury_DGS2_imputed  Core_CPI_CPILFESL_imputed  \
0                         1                          0   
1

# What’s Going On in the Data?

1. Macro Columns NaN Count
Most macro columns: 0 NaNs.

A handful (e.g., HY_Spread_BAMLH0A0HYM2, 2Y_Treasury_DGS2, etc.): 3–5 NaNs remain (out of 5114 rows).

This is trivial: It means 99.97% of the macro data is filled.

2. Missingness Indicator Mismatches
All zeros for mismatches:

the *_imputed columns perfectly reflect whether a value was missing in the original data.

This is ideal—no logic errors in tracking missingness.

3. Spot Checks
Forward-filling looks correct:

Imputed (filled) values appear as expected after a real observation (e.g., GDP filled for all days between quarterly releases).

Macro columns that were NaN at the start or have sporadic gaps: the indicator properly flags all imputed values.

4. Macro Coverage Summary
Mean non-NaN %: 99.97%

Median: 100%

This is excellent coverage for financial macro data.

5. Indicator Columns (First Rows)
The *_imputed columns are 1 when the original was missing, otherwise 0, just as designed.

In [15]:
total_mismatches = 0
for col in macro_cols:
    ind_col = f"{col}_imputed"
    mismatches = (df_orig[col].isna() != (df_macro[ind_col] == 1)).sum()
    total_mismatches += mismatches
    if mismatches:
        print(f"Mismatches for {col}: {mismatches}")
print(f"Total mismatches: {total_mismatches}")

Total mismatches: 0


# Fill the remaining NaNs 

Most ML models including KANS cannot handle NaN values, they will crash. 

Quant workflow best practice includes model stability, reproducibility, and full length time series. 

I'll do a foward fill then backward fill for standard practice here/

In [16]:
df_macro[macro_cols] = df_macro[macro_cols].bfill().ffill()
print(df_macro[macro_cols].isna().sum())  # Should now be all zeros

GDP_GDPC1                                 0
HY_Spread_BAMLH0A0HYM2                    0
M2_M2SL                                   0
2Y_Treasury_DGS2                          0
Core_CPI_CPILFESL                         0
Housing_Starts_HOUST                      0
10Y_Treasury_DGS10                        0
TED_Spread_TEDRATE                        0
CPI_CPIAUCSL                              0
PPI_PPIACO                                0
Economic_Policy_Uncertainty_USEPUINDXD    0
VXNCLS_VXNCLS                             0
BAA10Y_Spread_BAA10Y                      0
Payrolls_PAYEMS                           0
Industrial_Production_INDPRO              0
5Y_Treasury_DGS5                          0
Median_House_Price_MSPUS                  0
1Y_Treasury_DGS1                          0
Unemployment_UNRATE                       0
Fed_Funds_FEDFUNDS                        0
VIXCLS_VIXCLS                             0
Total_Loans_TOTLL                         0
30Y_Treasury_DGS30              

# Save

In [17]:
# Save your cleaned, imputed, feature-rich macro dataframe
save_path = r"C:\Users\manov\OneDrive\The Quant Desk\kan_mega_final_clean.parquet"
df_macro.to_parquet(save_path, index=False)
print(f"Saved to {save_path}!")

Saved to C:\Users\manov\OneDrive\The Quant Desk\kan_mega_final_clean.parquet!


In [18]:
"""
kan_mega_final_clean.parquet
----------------------------
- All macro and market columns forward-filled, time-interpolated, and boundary-filled.
- Missingness indicators for each macro variable (1 = imputed, 0 = observed).
- Ready for direct input into KAN or other ML models. No NaNs remain.
"""


'\nkan_mega_final_clean.parquet\n----------------------------\n- All macro and market columns forward-filled, time-interpolated, and boundary-filled.\n- Missingness indicators for each macro variable (1 = imputed, 0 = observed).\n- Ready for direct input into KAN or other ML models. No NaNs remain.\n'